In [ ]:
!date

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import scanpy as sc
import anndata

import subprocess
from scipy.io import mmread, mmwrite
from scipy.sparse import csr_matrix

In [ ]:
projdir = '/u/project/cluo/terencew/igvf/2023_YR2/multiome/'
donors = sorted(list(np.loadtxt(f'{projdir}/txt/donors.txt', dtype=str)))
tmp_order = ['Start', 'Sendai', 'Delayed', 'Fail1', 'Inter1',  'Inter2', 'Fail2', 'IPS']
clusts = [f'{x}_{y}' for x,y in zip(np.repeat(tmp_order, len(donors)), np.tile(donors, len(tmp_order)))]

In [ ]:
tf_families_path = f'{projdir}/csv/atac/chromvar/tf_families.tsv'
tf_families = pd.read_csv(tf_families_path, sep='\t')
tf_families.shape

In [ ]:
tf_families.head()

In [ ]:
tf_families[tf_families['subfamily'] == 'SP']

In [ ]:
tf_fam_dict = dict(zip(tf_families['column_name'], tf_families['family']))
tf_sub_dict = dict(zip(tf_families['column_name'], tf_families['subfamily']))

In [ ]:
adata = sc.read_h5ad(f'{projdir}/h5ad/atac/processed/chromvar/intersect_rna.h5ad')

In [ ]:
adata

In [ ]:
tfs = adata.var['TF']
tfs.shape

In [ ]:
zscore_list = []
for clust in tmp_order:
    tmp_adata = adata[adata.obs['coarse'] == clust]
    mean_zscore = tmp_adata.X.mean(axis=0)
    zscore_list.append(mean_zscore)
    

In [ ]:
mtx = np.vstack(zscore_list)
merged = pd.DataFrame(mtx, index=tmp_order, columns=tfs)
merged.shape

In [ ]:
merged.head()

In [ ]:
merged.mean(axis=1)

### clip extreme values

In [ ]:
tmp = merged.T['Fail1'].idxmax()
merged[tmp]

In [ ]:
merged_clipped = merged.clip(-10, 10)

In [ ]:
merged_clipped.mean(axis=1)

In [ ]:
merged_clipped.T.corr()

In [ ]:
sns.heatmap(merged_clipped, vmin=-2, vmax=2, cmap='coolwarm')

### subset to families

In [ ]:
sub_cols = pd.Index([tf_sub_dict.get(x) for x in merged_clipped.columns])
sub_cols.unique()

In [ ]:
tmp_clipped = merged_clipped.T
tmp_clipped['subfamily'] = sub_cols
final_clipped = tmp_clipped.groupby('subfamily').mean().T

In [ ]:
sns.heatmap(final_clipped, vmin=-2, vmax=2, cmap='coolwarm')

In [ ]:
corr_df = final_clipped.T.corr()
corr_df

In [ ]:
# merged_filt.shape

In [ ]:
out = f'{projdir}/csv/atac/chromvar/coarse.deviations.csv'
final_clipped.to_csv(out, sep='\t')

In [ ]:
outdir = '/u/project/cluo/terencew/igvf/2023_YR2/final_figures/csv/figure3'
final_clipped.to_csv(f'{outdir}/atac_chromvar_coarse_deviation_subfamily.csv', sep='\t')

### clusters?

In [ ]:
adata.obs['cluster']

In [ ]:
sub_clusters = list(np.loadtxt(f'{projdir}/txt/clusters/clusters.txt', dtype=str))

In [ ]:
zscore_list = []
for clust in sub_clusters:
    tmp_adata = adata[adata.obs['cluster'] == clust]
    mean_zscore = tmp_adata.X.mean(axis=0)
    zscore_list.append(mean_zscore)
    
mtx = np.vstack(zscore_list)
merged = pd.DataFrame(mtx, index=sub_clusters, columns=tfs)
merged.shape

In [ ]:
merged_clipped = merged.clip(-10, 10)
tmp_clipped = merged_clipped.T
tmp_clipped['subfamily'] = sub_cols
final_clipped = tmp_clipped.groupby('subfamily').mean().T
final_clipped.shape

In [ ]:
sns.heatmap(final_clipped, vmin=-2, vmax=2, cmap='coolwarm')

In [ ]:
corr_df = final_clipped.T.corr()
corr_df.head()

In [ ]:
sns.heatmap(corr_df, cmap='coolwarm')

In [ ]:
out = f'{projdir}/csv/atac/chromvar/cluster.deviations.csv'
final_clipped.to_csv(out, sep='\t')
outdir = '/u/project/cluo/terencew/igvf/2023_YR2/final_figures/csv/figure3'
final_clipped.to_csv(f'{outdir}/atac_chromvar_cluster_deviation_subfamily.csv', sep='\t')

In [ ]:
!date